# Puncturing a clot in 3-D

A thin needle pushed into a blood clot first dents it. Then, all at once, the clot tears open under the tip and the
needle goes in: the force on the needle drops. How hard you have to push before that happens, and how the tear runs,
matter to anyone who removes clots with a wire, a needle or a small robot.

You will build that experiment with MIME, a Python library for simulating microrobots, and its soft-tissue plugin,
`mime-softtissue`: a soft block that can tear, and a rigid tip pushed into it. You will run a small version live, plot
the force against depth, look at runs computed on bigger computers, and watch the clot tear in 3-D.

The clot's material is not made up: its stiffness and toughness were **calibrated** by this plugin against published
tearing tests on human whole-blood clot (Lohr et al. 2025, *Int. J. Fract.*; their raw data, shared CC0), in a
separate replication.

Run the cells one by one from the top (click a cell, then press Shift+Enter).

**Where this runs, and two settings that must come first.** JAX, the library MIME computes with, reads these before
it starts: float64 numbers (the tearing solve needs them), full precision for matrix products on a GPU, and one GPU
code path switched off (it gave wrong numbers on some GPUs). They change nothing on a CPU.

In [ ]:
import os, sys
in_colab = "google.colab" in sys.modules
os.environ.setdefault("XLA_FLAGS", "--xla_gpu_enable_triton_gemm=false")
os.environ.setdefault("JAX_DEFAULT_MATMUL_PRECISION", "highest")
os.environ.setdefault("JAX_ENABLE_X64", "1")
print("in Colab:", in_colab)

**Install (Colab only).** MIME at the exact commit this notebook was checked with, then the plugin from its public
repository, at the branch that carries this tutorial. Allow two or three minutes.

In [ ]:
# MIME's exact commit this notebook was last executed against (it holds the surfaces seam, #340). To update it: run
# the notebook end to end against a newer commit of MIME's release/v0.3.0, then put that commit's full sha here.
MIME_COMMIT = "99614e487c90f091e1fcbfa712aa14aa583634b5"
# The plugin's branch that carries this tutorial (its pull request is a draft; "main" once it is merged).
PLUGIN_REF = "tutorial-puncture-3d"
if in_colab:
    import subprocess

    def pip_install(spec):
        done = subprocess.run([sys.executable, "-m", "pip", "install", "-q", spec], capture_output=True, text=True)
        if done.returncode:
            raise RuntimeError("pip failed:\n" + done.stderr[-2000:])

    pip_install("mime-engine[verify] @ git+https://github.com/Microrobotics-Simulation-Framework/MIME@" + MIME_COMMIT)
    pip_install("mime-softtissue @ git+https://github.com/Microrobotics-Simulation-Framework/mime-softtissue.git@"
                + PLUGIN_REF)
    print("installed")

**Load the pieces we use.** Every number in MIME carries its unit: `mm` is millimetres, `s` seconds, `deg` degrees.
The plugin's pieces come in as `st` (soft tissue). The last lines keep what JAX compiles in a cache on disk, so running
a cell again is quicker.

In [ ]:
import json
import tempfile
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from mime import Body, Experiment, State, viz
from mime.assets import reference, resolve
from mime.dynamics import RigidInertial
from mime.scene import Scene, Z
from mime.things import Part
from mime.units import N, deg, kg, kg_m2, m2, m_per_s, m_per_s2, mm, rad_per_s, s

import mime_softtissue.entries as st
from mime_softtissue.studies import results as published

import jax
from mime.env import use_xla_cache, user_xla_cache_root

# Keep the programs JAX compiles on disk, so a cell run again (and the based_on and save cells below) compiles less;
# MIME's own folder for it, the one `mime run` uses. The numbers are the same with it and without it.
use_xla_cache(user_xla_cache_root())
jax.config.update("jax_persistent_cache_min_compile_time_secs", 0)
jax.config.update("jax_persistent_cache_min_entry_size_bytes", -1)
print("JAX computes on:", jax.devices())

**The settings.** The numbers you can change later, in "Try it yourself". The tip is a sharp cone, like the point of
a needle: its half-angle is how pointed it is, and its very apex is rounded off a little (a real point is never
perfectly sharp). The grid is how finely the clot is cut into small cells for the
computation: finer is more accurate and slower.

In [ ]:
cone_half_angle = 20.0  # degrees
apex_radius = 0.05      # millimetres: how much the very point is rounded
needle_radius = 1.0     # millimetres: the needle's shaft, behind the cone
push_to = 3.0           # millimetres: how deep the tip goes
grid = 0.125            # millimetres: the size of the clot's cells near the tip

**The clot's material, from a record.** MIME keeps measured and calibrated values in *records*: a named file with its
source, its licence and a check of its bytes, which an experiment names instead of copying numbers. This one is the
whole-blood clot calibrated by the plugin's tearing replication. Each value says where it comes from; one is a
placeholder (the clot's Poisson ratio, which tearing tests in thin sheets cannot measure).

In [ ]:
clot_record = "softtissue.whole_blood_clot"
with open(resolve(reference(clot_record))) as f:
    record = json.load(f)
for name, value in record["values"].items():
    print(f"{name:>18} = {value:<12.6g} {record['labels'][name][:96]}")

**Start the experiment.** An experiment holds the whole world. Here "up" is z and there is no gravity (it is tiny
against the forces of the needle). MIME moves time forward in steps of 0.2 seconds; the needle moves at 0.1 mm a second,
so it goes 0.02 mm deeper each step. Each step the clot is solved at rest under the needle (the push is slow, so the
clot's own inertia does not matter).

In [ ]:
experiment = Experiment("clot_puncture", scene=Scene.new(up=Z, units="m"), gravity=m_per_s2(0.0), dt=s(0.2))
speed = 0.1e-3  # metres a second

**The clot.** A cylinder of clot 10 mm across and 5 mm tall, its bottom held. The plugin's block solves how it
stretches and where it tears (a *phase field*: a damage value from 0, intact, to 1, broken, at every point). With
`dimension="axisymmetric"` it computes one slice and assumes the rest is the same all the way round, which is fast;
`dimension="3d"` computes the real 3-D block, later on. The block must be told which tip shape will press on it.

In [ ]:
clot = experiment.add_body(Body(
    "clot",
    dynamics=st.GelBlock(radius=mm(5.0), height=mm(5.0), material=clot_record, h=mm(grid), fine=mm(1.0),
                         dimension="axisymmetric", tip_shape="softtissue.cone_tip"),
    initial=State(position=mm((0.0, 0.0, 0.0))),
))

**The needle.** A rigid body (it does not bend) whose shape is the plugin's cone tip: pointed at the bottom, its apex
rounded off by 0.05 mm, widening to the needle's 2 mm shaft (which it reaches 2.6 mm up from the point). It starts with
its point just touching the top of the clot, 5 mm up, and moving down.

In [ ]:
needle = st.ConeTip(radius=mm(needle_radius), tip_radius=mm(apex_radius), half_angle=deg(cone_half_angle),
                    length=mm(6.0))
tip = experiment.add_body(Body(
    "tip",
    dynamics=RigidInertial(),
    parts={"tip": Part(needle)},
    initial=State(position=mm((0.0, 0.0, 5.0)), orientation=(1.0, 0.0, 0.0, 0.0),
                  velocity=m_per_s((0.0, 0.0, -speed)), angular_velocity=rad_per_s((0.0, 0.0, 0.0))),
))
experiment.declare("tip.mass", kg(1.2e-5), reason="a steel needle of a few millimetres (its mass does not matter here)")
experiment.declare("tip.inertia", kg_m2((1e-11, 1e-11, 1e-12)), reason="the same needle's order")

Now the physics, one piece per cell. If you run one of these cells twice, its piece is added twice and MIME will
refuse to build. To fix it, run the cells again from "Start the experiment" (or use Runtime, then Run all).

**The drive.** A servo pushes the needle down at a steady 0.1 mm a second for the whole run. The run has one step
more than the depth needs (the clot sees where the needle was at the end of the step before), rounded up to a whole
number of 6-step frames for the view.

In [ ]:
steps = 6 * int(np.ceil((push_to / 0.02 + 1) / 6))
experiment.attach(st.Servo(speed=m_per_s(speed), insert_steps=steps), body="tip")

**The contact.** Where the needle's surface meets the clot, the clot is pushed out of the way: a very stiff spring on
every point of the clot that the needle's shape would overlap. No friction and no stickiness here (the clot's own
friction and adhesion are not measured). The spring's stiffness is a numerical choice, 100 times the clot's stiffness
over the cell size.

In [ ]:
young = record["values"]["young_modulus"]
experiment.attach(st.TipContact(friction=0.0, adhesion=0.0, stiffness=100.0 * young / (grid * 1e-3)),
                  tip="tip", solid="clot")

**What to measure.** The force the clot puts on the needle and the needle's depth, every step; and the size of the
tear (the area of the crack).

In [ ]:
force_probe = experiment.observe(st.TipForce(solid="clot"))
crack_probe = experiment.observe(st.CrackSize(solid="clot"))

**Run it.** MIME checks the experiment, builds it into fast code, and pushes the needle just over 3 mm into the clot in
156 small steps, keeping the two measurements at every step (`record=`) and a 3-D frame every sixth step for the view
(`stage_every=`), in a new folder each time the cell runs. On
a laptop's processor this took about three minutes (one core, the compile included); Colab's free CPU may take
longer.

In [ ]:
t0 = time.time()
built = experiment.build()
clot_run = Path(tempfile.mkdtemp(prefix="clot_")) / "run"  # a fresh folder each time this cell runs
run = built.run_to(clot_run, s(steps * 0.2), record=[force_probe, crack_probe], stage_every=s(1.2))
depth = run.result[force_probe].depth.values.to(mm)
force = run.result[force_probe].force.values.to(N) * 1e3  # millinewtons
crack = run.result[crack_probe].area.values.to(m2) * 1e6  # square millimetres
first_drop = next((k for k in range(1, len(force)) if force[k] < 0.97 * force[:k].max()), None)
print(f"Took {time.time() - t0:.0f} s. At {depth[-1]:.2f} mm deep the force on the needle is {force[-1]:.2f} mN.")
if first_drop is not None:
    print(f"It first dropped (by more than 3 %) at {depth[first_drop]:.2f} mm, from {force[:first_drop].max():.2f} mN.")

**Plot the push.** The force against the needle's depth, and the crack's area. Matplotlib, which Colab has, draws it.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(depth, force, color="#2a78d6", linewidth=2)
ax[0].set_xlabel("depth of the tip (mm)"); ax[0].set_ylabel("force on the tip (mN)")
ax[0].set_title("The push: the needle into the clot")
ax[1].plot(depth, crack, color="#d6612a", linewidth=2)
ax[1].set_xlabel("depth of the tip (mm)"); ax[1].set_ylabel("crack area (mm²)")
ax[1].set_title("The tear")
for a in ax:
    a.grid(color="#e1e0d9")
plt.tight_layout(); plt.show()

**Watch it in 3-D.** The clot coloured by damage (dark: intact, yellow: torn) and the needle going in. The block was
computed as one slice; the view turns that slice three quarters of the way round, so you can see inside. Drag to turn
the view, scroll to zoom, and use the play button or the slider. The view needs an internet connection.

In [ ]:
viz.display(clot_run, caption="The axisymmetric clot, revolved for display, coloured by damage (0 to 1).")

## In real 3-D

A needle's tear is not the same all the way round: a sharp tip opens a flat crack, a slit, which an axisymmetric slice
cannot hold. The plugin's 3-D block computes **half** of the real block (the other half is its mirror image across a
plane through the needle), with a fast solver for the large systems that 3-D brings (a multigrid, low precision where
it is safe, inner iterations stopped early, and a step that does not settle tried again in smaller steps). The needle
sits 0.05 mm off the centre line: a real needle is never exactly centred, and the slit needs a direction to open in.
Why half and not a quarter: with the needle a fifth of a cell off the line, the half block gives the whole block's force
to 4e-7, while a quarter block reads 55 % high at 1 mm deep for this sharp cone, most likely because its two planes of
symmetry pin the material on the needle's axis (the plugin's engine, the three blocks run on 0.177 mm cells).

Changing to 3-D is a few settings on the clot (`dimension="3d"`, `symmetry="half"`, the cells near the tip) and the
needle placed a little off the line. It costs more: 0.4 mm of push takes about 6 minutes on a 2-core computer, the
compile included, and the 1.5 mm run below about half an hour, so the runs below were made on GitHub's computers
(`examples/puncture.py`) and are loaded here; the cell can run its first 0.4 mm live. The next cell
makes the change the way you would make your own: a new experiment **based on** the one above, which the plugin ships
saved, with the clot, the needle and the pieces that depend on them declared again (MIME replaces each one taken, and
says so).

In [ ]:
from mime_softtissue.studies.results import saved

grid_3d = 0.25  # millimetres: the 3-D block's cells near the tip
experiment_3d = Experiment.based_on(saved("clot_puncture"), name="clot_puncture_3d")
experiment_3d.add_body(Body(
    "clot",
    dynamics=st.GelBlock(radius=mm(5.0), height=mm(5.0), material=clot_record, h=mm(grid_3d), fine=mm(2.0),
                         fine_across=mm(0.8), dimension="3d", symmetry="half", tip_shape="softtissue.cone_tip"),
    initial=State(position=mm((0.0, 0.0, 0.0))),
))
# The saved experiment's step (0.2 s) is kept; at twice the speed each step goes 0.04 mm deeper. The model is
# quasi-static, so the speed sets only the clock.
speed_3d = 2 * speed
experiment_3d.add_body(Body(
    "tip",
    dynamics=RigidInertial(),
    parts={"tip": Part(needle)},
    initial=State(position=mm((0.2 * grid_3d, 0.0, 5.0)), orientation=(1.0, 0.0, 0.0, 0.0),
                  velocity=m_per_s((0.0, 0.0, -speed_3d)), angular_velocity=rad_per_s((0.0, 0.0, 0.0))),
))
experiment_3d.attach(st.Servo(speed=m_per_s(speed_3d), insert_steps=40), body="tip")
experiment_3d.attach(st.TipContact(friction=0.0, adhesion=0.0, stiffness=100.0 * young / (grid_3d * 1e-3)),
                     tip="tip", solid="clot")
for line in experiment_3d.plan().replaced:
    print("replaced:", line[:150])

# True to push it 0.4 mm in (10 steps): about 6 minutes on a 2-core GitHub runner, the compile included; Colab's
# free CPU may take twice that.
run_3d_live = False
if run_3d_live:
    t0 = time.time()
    run_3d = experiment_3d.build().run_to(
        Path(tempfile.mkdtemp(prefix="clot3d_")) / "run", s(2.0), stage_every=s(0.4),
        record=["softtissue.tip_force:clot.force", "softtissue.tip_force:clot.depth", "softtissue.crack_size:clot.area"])
    series = run_3d.result["softtissue.tip_force:clot"]
    depth_3d, force_3d = series.depth.values.to(mm), series.force.values.to(N) * 1e3
    shipped = published.puncture()["runs"]["c3d-clot-h0.25"]
    k = int(np.argmin(np.abs(np.array(shipped["depth_mm"]) - depth_3d[-1])))
    print(f"Took {time.time() - t0:.0f} s. At {depth_3d[-1]:.2f} mm the force on the needle is {force_3d[-1]:.4f} mN "
          f"(the run shipped with the plugin, the same cells: {shipped['force_N'][k] * 1e3:.4f} mN at "
          f"{shipped['depth_mm'][k]:.2f} mm).")
    viz.display(run_3d.outputs.parent, caption="Your 3-D run: the computed half of the block, coloured by damage.")

**The converged runs.** Computed on GitHub's runners with the same experiment (`examples/puncture.py`), and saved in the
plugin as small files.

- **Left: the cone into the clot, axisymmetric, at three cell sizes.** The live run above used the coarsest. Where the
  curves differ, the coarse grid is not converged: that difference is the live run's error.
- **Right: the same cone in 3-D**, through MIME (`dimension="3d"`, half the block) at the cell sizes that ran, and the
  same 3-D solver run on its own by the plugin's engine work where those runs are shipped. The finest axisymmetric curve
  is drawn again, dotted. Read the gap with care: the 3-D curve sits below from the first touch, before any crack
  forms (its crack area is zero until about 0.5 mm), so at these cells much of the gap is how a coarse 3-D grid feels a
  sharp tip, not the slit. By 1.5 mm the engine's own runs agree between 0.177 and 0.125 mm cells (to 0.6 %), at about
  16 % below the axisymmetric force; but computing the contact inside the cells instead of at the grid's points gave a
  3-D force above the axisymmetric one on 0.25 mm cells (the engine work, measured). So the 3-D curves show the 3-D build
  working; they are not yet a measurement of what the slit changes.

In [ ]:
from mime_softtissue.studies import puncture3d_results

P = published.puncture()
print(P["about"])
runs = P["runs"]
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
axi = ("cone-clot-h0.125", "#9bbbe0"), ("cone-clot-h0.0625", "#2a78d6"), ("cone-clot-h0.03125", "#0b3a75")
for name, colour in axi:
    if name in runs:
        r = runs[name]
        ax[0].plot(r["depth_mm"], np.array(r["force_N"]) * 1e3, color=colour, label=f"cells {r['h_mm']} mm")
ax[0].plot(depth, force, ":", color="k", label="live, this notebook")
ax[0].set_title("20-degree cone into whole-blood clot (axisymmetric)")
finest = next((runs[n] for n, _ in reversed(axi) if n in runs), None)
if finest is not None:
    ax[1].plot(finest["depth_mm"], np.array(finest["force_N"]) * 1e3, ":", color="0.4",
               label=f"axisymmetric, cells {finest['h_mm']} mm")
for name, colour in (("c3d-clot-h0.25", "#f2a65a"), ("c3d-clot-h0.177", "#d6612a")):
    if name in runs:
        r = runs[name]
        ax[1].plot(r["depth_mm"], np.array(r["force_N"]) * 1e3, color=colour,
                   label=f"3-D through MIME, cells {r['h_mm']} mm")
for name in puncture3d_results.names():
    if not name.startswith("clot_cone"):  # the symmetry check's runs used an earlier set of the clot's constants
        continue
    shipped = puncture3d_results.load(name)
    meta = shipped.meta
    if (meta["tip"]["kind"] != "cone" or meta["block"]["symmetry"] != "half" or name.endswith("live")
            or meta["settings"].get("fracture", "phase_field") != "phase_field"):
        continue
    ax[1].plot(np.array(shipped.rows["depth_m"]) * 1e3, np.array(shipped.rows["force_N"]) * 1e3, "--", color="#8a2be2",
               alpha=float(np.clip(0.4 + 0.6 * (0.25e-3 - meta["block"]["h"]) / 0.125e-3, 0.2, 1.0)),
               label=f"3-D engine alone, cells {meta['block']['h'] * 1e3:.3g} mm")
ax[1].set_title("The same cone in 3-D (half the block)")
for a in ax:
    a.set_xlabel("depth of the tip (mm)"); a.set_ylabel("force on the tip (mN)")
    a.grid(color="#e1e0d9"); a.legend(fontsize=8)
plt.tight_layout(); plt.show()

**A check of the 3-D block.** A 1 mm flat punch into a placeholder gel, the case the fast 3-D solver was tuned on. MIME's
block with the tuned solver's own arithmetic (`solver="all2"`) should lie on that solver's earlier runs outside MIME:
that is the check that MIME's version is the solver that was tested. The block's default (`solver="tip3d"`) differs from
it in three ways: the average volume change of each cell (for nearly incompressible material), the damage held between
0 and 1 exactly, and an accelerated iteration with a limit on its passes. It reads 3 to 5 % lower before the peak and
peaks earlier and lower (8.9 against 10.5 mN), the same at 0.2 and 0.1 mm cells. Switching the three one at a time
shows that the first alone makes the gap: the default with the tuned solver's volume treatment gives the tuned
solver's forces to four digits, and the other two change nothing measurable here.

**What the peaks say: neither is converged.** The tuned arithmetic (full integration, as in the solver outside MIME)
holds each cell's volume change at each of its eight integration points; the default holds it once a cell, on the cell's
average (mean dilatation).

- **Both plotted peaks are under-resolved** (cells two and one times the tear's length scale, 0.1 mm); neither is the
  converged puncture force.
- **Refined, both element choices meet.** An axisymmetric build of this punch, refined to cells of 0.35 to 0.25 times
  the length scale, converges to about 11.5 mN with both element choices (within 4 %). Against it the plotted 3-D peaks
  are under-resolved: about 23 % low for mean dilatation (the default) and 8 to 10 % low for full integration at
  0.1 mm cells. The default's 3-D peak still rises with refinement (8.9, 9.3, 10.1 mN at 0.1, 0.07, 0.05 mm cells).
- **The two elements' elastic forces match** (both within about 1 % of the classical elastic solution for a sphere
  pressed into the gel, Hertz's).
- **Full integration does lock** where the material's bulk-to-shear ratio is high, as in a strongly damaged zone (the
  tearing law softens shear but never compression): on Hertz's case it read 1.6 times the elastic force with a bulk
  modulus of 500 shear moduli. On the peak that moves it little (-1 to +4 % on the fine axisymmetric grids); its real
  effect is after the peak, where full integration holds 8 to 10 mN past 0.36 mm while mean dilatation falls.
- **The tuned arithmetic agreeing with the solver outside MIME** is the same arithmetic: it shows that MIME runs that
  solver, not that the solver's peak is right.

Not checked: the default element's pressure patterns inside the damaged zone, and where the force after the peak settles
(the default's is still grid-dependent there).

In [ ]:
ref = P["reference"]
plt.figure(figsize=(7, 4.5))
full = "full integration"
plt.plot(ref["baseline"]["depth_mm"], np.array(ref["baseline"]["force_N"]) * 1e3, "s", mfc="none", color="0.5",
         label=f"outside MIME, fixed steps, {full}")
plt.plot(ref["all2"]["depth_mm"], np.array(ref["all2"]["force_N"]) * 1e3, "^", mfc="none", color="0.2",
         label=f"outside MIME, fast configuration, {full}")
punch = (("p3d-gel-h0.1-all2", "#2a78d6", "-", full),
         ("p3d-gel-h0.2-tip3d", "#f2a65a", "--", "mean dilatation (the default)"),
         ("p3d-gel-h0.1-tip3d", "#d6612a", "--", "mean dilatation (the default)"))
for name, colour, style, element in punch:
    if name in runs:
        r = runs[name]
        plt.plot(r["depth_mm"], np.array(r["force_N"]) * 1e3, style, color=colour,
                 label=f"through MIME, solver {r['solver']}, cells {r['h_mm']} mm, {element}")
plt.xlabel("depth of the punch (mm)"); plt.ylabel("force on the punch (mN)")
plt.title("1 mm flat punch into a placeholder gel (3-D): under-resolved peaks"); plt.grid(color="#e1e0d9")
plt.legend(fontsize=7)
plt.show()
for name, _, _, element in punch:
    if name in runs:
        r = runs[name]
        k = int(np.argmax(r["force_N"]))
        print(f"solver {r['solver']}, cells {r['h_mm']} mm ({element}): "
              f"peak {r['force_N'][k] * 1e3:.2f} mN at {r['depth_mm'][k]:.2f} mm (not converged)")
for name in ("p3d-clot-h0.1-tip3d",):
    if name in runs:
        r = runs[name]
        k = int(np.argmax(r["force_N"]))
        print(f"The same punch into the clot (3-D, a prediction): largest force {r['force_N'][k] * 1e3:.2f} mN at "
              f"{r['depth_mm'][k]:.2f} mm (the run reached {r['depth_mm'][-1]:.2f} mm).")

**Watch the 3-D block tear.** The half block that was computed, coloured by damage, and the needle. The flat face at the
back is the plane of symmetry: the slit inside the block shows on it.

In [ ]:
ANIMATED = "c3d-clot-h0.25"
viz.display(published.puncture_run(ANIMATED),
            caption="The 20-degree cone into the clot in 3-D: the computed half of the block, coloured by damage (0 to 1).")

## Why a phase field: deleting cells has no answer to converge to

A common way to let a simulated material tear is **element deletion**: a cell stretched past a limit is removed. Under
a sharp tip that has a known problem. The finer the grid, the more sharply the stretch peaks at the point, so the limit
is reached sooner and the computed puncture force keeps falling as the cells shrink: refining never settles it. The
phase field spreads the crack over a set width instead (its length scale, 0.25 mm for this clot, from the tearing
calibration), so as the cells shrink its force can settle.

The plugin's 3-D engine ran both on the same 20-degree cone into the same clot (half the block, the grid's points
carrying the contact) at three cell sizes. Element deletion removes a cell once it is stretched 1.55 times, the stretch
at which the clot's sheets broke in the tearing tests. These runs are loaded, not run here: MIME's 3-D block offers only
the phase field.

In [ ]:
pf, dl = {}, {}
for name in puncture3d_results.names():
    if not name.startswith("clot_cone"):  # the same clot, cone and constants in every run compared
        continue
    shipped = puncture3d_results.load(name)
    meta = shipped.meta
    if meta["tip"]["kind"] != "cone" or meta["block"]["symmetry"] != "half" or name.endswith("live"):
        continue
    if meta["settings"].get("contact_at", "nodes") != "nodes":
        continue
    kind = meta["settings"].get("fracture", "phase_field")
    (dl if kind == "deletion" else pf)[round(meta["block"]["h"] * 1e3, 4)] = shipped
cells_mm = sorted(set(pf) & set(dl), reverse=True)
shades = dict(zip(cells_mm, ("#9bbbe0", "#2a78d6", "#0b3a75", "#06203f")))
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
first = {}
for h in cells_mm:
    for runs_by_cell, style, label in ((pf, "-", "phase field"), (dl, "--", "element deletion")):
        r = runs_by_cell[h]
        ax[0].plot(np.array(r.rows["depth_m"]) * 1e3, np.array(r.rows["force_N"]) * 1e3, style, color=shades[h],
                   label=f"{label}, cells {h:g} mm")
    peak, at, found = dl[h].puncture()
    first[h] = (peak, at) if found else None
    if found:
        ax[0].plot(at * 1e3, peak * 1e3, "x", color=shades[h], ms=10, mew=2)
ax[0].set_xlabel("depth of the tip (mm)"); ax[0].set_ylabel("force on the tip (mN)")
ax[0].set_title("The cone into the clot in 3-D (x: deletion's first puncture)")
at_depth = {d: [float(np.interp(d * 1e-3, pf[h].rows["depth_m"], pf[h].rows["force_N"])) * 1e3 for h in cells_mm]
            for d in (1.0, 1.5)}
ax[1].plot(cells_mm, [first[h][0] * 1e3 if first[h] else np.nan for h in cells_mm], "x--", color="#d6612a", ms=9,
           mew=2, label="element deletion: its first puncture force")
for d, mark in ((1.5, "o-"), (1.0, "s:")):
    ax[1].plot(cells_mm, at_depth[d], mark, color="#2a78d6", label=f"phase field: the force at {d:g} mm deep")
ax[1].invert_xaxis(); ax[1].set_ylim(bottom=0)
ax[1].set_xticks(cells_mm); ax[1].set_xticklabels([f"{h:g}" for h in cells_mm])
ax[1].set_xlabel("cell size near the tip (mm), finer to the right"); ax[1].set_ylabel("force (mN)")
ax[1].set_title("What each method reports as the cells shrink")
for a in ax:
    a.grid(color="#e1e0d9"); a.legend(fontsize=8)
plt.tight_layout(); plt.show()
for h in cells_mm:
    text = (f"first puncture {first[h][0] * 1e3:.3f} mN at {first[h][1] * 1e3:.2f} mm" if first[h]
            else "no puncture in the run")
    print(f"cells {h:g} mm: element deletion {text}; phase field {at_depth[1.0][cells_mm.index(h)]:.3f} mN at 1 mm, "
          f"{at_depth[1.5][cells_mm.index(h)]:.3f} mN at 1.5 mm")

**Reading it.** Deletion's first puncture force about halves with each refinement, and the depth at which it comes
moves up with it: no grid gives its answer. The phase field's force at 1.5 mm deep settles (the two finest grids agree
to within 1 %); at 1 mm it is still rising, so it is not converged there. The two numbers on the right are not the same
quantity: the phase field had not punctured by 1.5 mm, so its force at a fixed depth is shown, not a puncture force.
What the comparison shows is the trend, which is the reason this plugin tears by a phase field.

## Viscoelasticity (a prediction)

Clot relaxes: pushed and held, its force falls over seconds to minutes, and pushed faster it is stiffer. The plugin's
viscoelastic option (branch `engine-visco`, `studies/visco_clot.py`; not yet part of this tutorial's block, so no cell
here runs it) adds two relaxing branches to the clot's law, calibrated on public ramp-and-hold tests of clot sheets
(CC0). Run axisymmetrically with the cone above, it **predicts**:

- **(a)** the puncture force rises with insertion speed, 12 to 21 % above the elastic model's from 0.1 to 10 mm/s, and is
  nearly flat above about 1 mm/s;
- **(b)** after pushing 1 mm in at 1 mm/s and holding, the force on the needle relaxes by 36 to 48 % over 10 to 300 s.

**Its limits, plainly.** One loading rate cannot pin the fast relaxation time, so the instantaneous stiffness is
uncertain by a factor of 1.8 to 4.6, and (a) could be anywhere from +13 to +43 %. The model relaxes more than the data
it was fitted to (the data: 21 to 24 % in 300 s). The branches' weights depend on how far the clot was stretched, which
the model does not. The held-out checks are mixed. All of it is a prediction, not validated.

## A published finding to match

Fakhouri, Hutchens and Crosby (2015, *Soft Matter* 11, 4723) pushed **sphere-tipped needles** of radius R from about
0.1 to 5 mm into a soft gel and measured the force at which it punctures, Fc. Fregonese and Bacca (2021, *J. Mech. Phys.
Solids*; arXiv 2412.09809) put those data on one curve: the force made dimensionless, Fc / (mu R²), against
mu R / Gc, where mu is the gel's shear modulus (7.2 kPa, as their curve takes it) and Gc its toughness (25 J/m²); its
theory needs a strongly
stiffening material (an Ogden exponent alpha of 9) to reach the data where needles are small.

The points below are **digitised** from their Fig. 5 (Exp. 1: Fakhouri's gel, with its error bars) and so are their
theory curves; only the numbers are used here, cited. The gel is a record too, `softtissue.puncture_gel`; its
phase-field length scale is a declared placeholder, since the paper states none and it sets the model's strength.

**The model.** The same experiment with a sphere-tipped needle (`st.PunchTip` with `fillet` equal to `radius`), and
MIME's `Study`, which runs one experiment at many settings: here the needle's radius, R = 0.5, 1, 2 and 4 mm, all
compiled once, at two cell sizes near the tip (0.2 and 0.1 mm; at 0.1 mm each radius ran as its own job). The gel's
toughness is held, as in the experiment (a record's values are not a study's axes). These are axisymmetric runs: a
ring-shaped crack is the only kind the axisymmetric block can form, while the gel's real crack is flat, so these points
are a first look, not the replication. They fall below the data, by about 2 times for the largest needle and up to 8
times for the smallest: the model's curve is flatter than the data's.

**Why they sit low: checked, not settled.** The model's elastic part is clean: the gel's shear modulus enters as the
paper's, with no stray factor, and a sphere pressed into the gel meets Hertz's elastic solution within 1 % on the
finest grid. The neo-Hookean points are converged in the grid and, for the small needles, insensitive to the length
scale: a finer grid still (0.05 mm cells, run separately, not plotted) moved the 0.5 and 1 mm needles' points by 1 to
2 % (the 0.5 mm one's with a smaller length scale too), and a length scale five times smaller moved the 0.5 mm needle's
by 4 % (it moves the large needles' more, about 40 % for the 2 mm one, not enough to close their gap). So these points
are what this law gives: 2 to 8 times below the data, and flatter.

The compiled curve uses a modulus that, in the source (Fakhouri, Hutchens and Crosby, *Soft Matter* 2015,
doi:10.1039/C5SM00230C), belongs to a different gel; the tested gel's is about half, which moves the data and widens
the miss. The source also reports that puncture is limited by crack nucleation, at 5 to 11 times the notched toughness
used here. A strongly stiffening law (Gent's, with its stiffening limit refitted to the theory's Ogden alpha = 9)
closes most of the gap, its points falling with about the data's slope, but they are not converged in the grid, and
they are steep in its stiffening limit and in the length scale, neither of which any measurement fixes. That is a
hypothesis, not a prediction, so it is not plotted. One more thing is unmeasured: the needle runs (both laws) use full
integration, which the punch check above measured locking (reading too stiff) at the high bulk-to-shear ratios of a
strongly damaged zone, and how much that changes the needle points is not known. Locking makes the model stiffer, so
it cannot explain points that sit too low, but it moves every point's distance to the bars. Strain stiffening and
nucleation may both be needed; both are being tested.

**The replication** is the 3-D block (half of it, the needle a little off the centre line) run by the plugin's 3-D
engine on its own, the same solver MIME's block runs: Fakhouri's gel at three values of mu R / Gc, the needle's radius
fixed and the toughness set by each value (a record's toughness is not a study's axis, so these ran outside
`Experiment`). For both, a filled marker is a run that punctured (its force's first drop); an open one is the largest
force reached before the run ended, a lower bound. The 3-D points (neo-Hookean, like the axisymmetric ones) sit 2.5 to
3.2 times under the bars' lower edges, none inside, and they are not converged either: at mu R / Gc = 0.62 the peak
rose with each refinement (Fc / (mu R²) 3.7, 4.4, then at least 5.0 on cells of R/4, R/5.7 and R/8; only the first is
shipped and plotted), and an extrapolation of that ladder, about 6.9, still sits 1.7 times under the bar. So computing
the flat crack in 3-D does not close the gap.

In [ ]:
F = published.fregonese_fig5()
mu, gc = 7200.0, 25.0
e1 = F["exp1"]
x = np.array([p["x"] for p in e1]); y = np.array([p["y"] for p in e1])
low = np.array([p["y_low"] for p in e1]); high = np.array([p["y_high"] for p in e1])
plt.figure(figsize=(8, 5))
plt.errorbar(x, y, yerr=[y - low, high - y], fmt="o", color="#2a78d6", mfc="none", capsize=3,
             label="Fakhouri et al. 2015, digitised from Fregonese and Bacca's Fig. 5")
for name, style in (("alpha2", "-"), ("alpha9", "-.")):
    c = np.array(F["theory"][name])
    keep = (c[:, 0] > 5e-3) & (c[:, 0] < 5)
    plt.plot(c[keep, 0], c[keep, 1], style, color="0.45", label=f"their theory, Ogden alpha = {name[5:]}")
markers = {"sweep-gel-h0.2": ("s", "#d6612a", "cells 0.2 mm"), "sweep-gel-h0.1": ("D", "#8a2be2", "cells 0.1 mm")}


def first_drop(force):  # the puncture: the first maximum the force then falls 2 % below; else the largest force
    for k in range(1, len(force) - 1):
        if force[k] >= force[k - 1] and force[k] >= force[k + 1] and min(force[k + 1:]) < 0.98 * force[k]:
            return force[k], True
    return max(force), False


for name, sweep in P.get("sweeps", {}).items():
    m, colour, label = markers.get(name, ("x", "k", name))
    for i, row in enumerate(sweep["members"]):
        R, (Fc, punctured) = row["R_mm"] * 1e-3, first_drop(row["force_N"])
        plt.plot(mu * R / gc, Fc / (mu * R ** 2), m, color=colour, ms=8, mfc=colour if punctured else "none",
                 label=f"this model, axisymmetric, {label}" if i == 0 else None)
for name in puncture3d_results.names():
    shipped = puncture3d_results.load(name)
    meta = shipped.meta
    tip_3d, mat = meta["tip"], meta["material"]
    if tip_3d["kind"] != "punch" or abs(tip_3d["fillet"] - tip_3d["radius"]) > 1e-9 or abs(mat["mu"] - mu) > 1.0:
        continue
    if meta["settings"].get("fracture", "phase_field") != "phase_field":
        continue
    R3, (Fc3, _, punctured) = tip_3d["radius"], shipped.puncture()
    plt.plot(mu * R3 / meta["gc_physical"], Fc3 / (mu * R3 ** 2), "*", ms=13, color="#0b3a75",
             mfc="#0b3a75" if punctured else "none",
             label=f"this model in 3-D, cells {meta['block']['h'] * 1e3:.3g} mm ({name})")
plt.xscale("log"); plt.yscale("log")
plt.xlabel("mu R / Gc"); plt.ylabel("Fc / (mu R²)")
plt.title("Puncture of a soft gel by sphere-tipped needles")
plt.grid(color="#e1e0d9", which="both"); plt.legend(fontsize=8); plt.show()

## What this tutorial claims

**It claims** that every experiment here builds and runs as written through MIME's `Experiment` and `Study`, and
that the outputs shown are what those runs gave; and that MIME's 3-D block is the plugin's 3-D engine, unchanged: run
through MIME and called directly at the same depths, on a coarse half block with the tip off the centre line, the forces
agree to 5e-13 (a 1 % change in the material's stiffness, planted, is caught at every step), and at the tuned solver's
own arithmetic (`solver="all2"`) the engine gives the forces of the solver it was tuned from, bit for bit. That is a
check of the code, not of the physics.

**The clot's material** is a set of three constants (Yeoh C1 363 ± 18 Pa, C2 434 ± 95 Pa, Gc 0.83 ± 0.17 J/m² at a
length scale of 0.25 mm), fitted on the raw data of Lohr et al. (2025, CC0). Its fit's own claim: at that Gc, the model
gives held-out group 5's peak force within 2 % (inside its band) and groups 1-3's 19 to 24 % low, for in-vitro whole-blood
clot in pure shear at 0.2 mm/s; it does not reproduce the notch dependence with one Gc. It holds in pure shear to a
stretch of about 1.5 at 0.2 mm/s; **puncture with these constants is a prediction.**

**It does not claim:**

- **A predicted puncture force for clot.** The clot's numbers were calibrated in tearing tests on thin sheets, not in
  puncture, and no needle-puncture measurement on clot with public numbers was found to check them against. The
  closest, guidewire push forces of about 15 to 56 mN through clot analogues (Reymond et al. 2024, read from its
  abstract only), is a different tip and test. The clot runs are **predictions**, labelled as such.
- **A converged live run.** The live run's cells (0.125 mm) give forces about 12 % above the finest run's at 1 mm
  deep, and within 5 % from 1.5 mm on (left plot above).
- **A converged 3-D run.** The 3-D runs here have cells of 0.25 mm (and 0.177 mm where shipped). A sharp tip that falls
  between the grid's points is not felt: the 3-D force reads almost nothing for the first 0.3 mm. How the needle's
  contact is computed (at the grid's points, as here, or inside the cells) changes the 3-D force by about a factor of
  two at these cells (the plugin's engine work, measured), which is more than the gap between the 3-D and the
  axisymmetric curves. So neither the 3-D numbers nor the slit's effect on them are claimed.
- **Forces after a peak, in 3-D.** MIME's 3-D block takes fixed steps of 0.04 mm. On the flat punch they hold the tuned
  solver's forces to within 0.5 % up to the peak and fall up to 11 % below them after it, where the tuned solver takes
  smaller steps (MIME's experiments take a fixed step). Neither of the punch's plotted peaks is converged: an
  axisymmetric build refined further converges to about 11.5 mN with both element choices, so at 0.1 mm cells the 3-D
  peaks are about 23 % (the default) and 8 to 10 % (full integration) low; after the peak, locking holds full
  integration's force up (see the punch's plot).
- **A needle's tear from an axisymmetric run.** It can only form ring-shaped cracks; a sharp tip's flat crack needs the
  3-D block.
- **The replication, yet.** The gel's master curve is the published finding to meet. The axisymmetric points (cells
  of 0.2 and 0.1 mm) sit below the data, by about 2 times for the 4 mm needle and up to 8 times for the 0.5 mm one,
  none inside its error bars, and at 0.4 to 0.6 of Fregonese and Bacca's alpha = 2 theory. The two cell sizes agree to
  3 % for the 1, 2 and 4 mm needles and differ by 23 % for the 0.5 mm one; a finer grid run separately moved the small
  needles' points by 1 to 2 %. What was checked: the convention and Hertz's solution, both clean; the neo-Hookean points
  converged in the grid and insensitive to the length scale at small needles, 2 to 8 times below the data and flatter.
  The compiled curve's modulus belongs, in the source, to a different gel (the tested gel's is about half, which
  widens the miss), and the source reports puncture limited by crack nucleation, at 5 to 11 times the notched toughness
  used here. A strongly stiffening law (Gent's) closes most of the gap, but its points are not converged in the grid
  and are steep in its stiffening limit and the length scale, which no measurement fixes: a hypothesis, not a
  prediction. Strain stiffening and nucleation may both be needed; both are being tested.
  The needle runs use full integration, which locks at high bulk-to-shear ratios (the punch check); its effect on them
  is unmeasured.
  The 3-D points (the engine's runs, neo-Hookean, not converged in the grid) sit about 3 times below the bars as well.

**What the model lacks:**

- **Rate.** Clot is viscoelastic (it relaxes over seconds to a minute) and puncture forces rise with needle speed; every
  run here is elastic and quasi-static (the viscoelastic option's predictions are in "Viscoelasticity", above).
- **Friction and adhesion** between the needle and the clot: set to zero here; the contact law has both, unmeasured.
- **Heterogeneity.** One uniform material; real clots have fibrin-rich and red-cell-rich regions, and fluid (about half
  their volume), which the model leaves out.
- **Two declared numbers.** The clot's Poisson ratio (0.45, a placeholder for the solver), and the phase field's length
  scale (0.25 mm for clot, from the tearing calibration; a placeholder for the gel), which sets the model's strength.
  The engine's own 3-D runs make the clot nearly incompressible (a bulk modulus of 50 shear moduli, as the fit assumed)
  where the record's 0.45 gives 9.7: MIME's 3-D runs at the same cells read 7 to 12 % below the engine's.

## Try it yourself

Change a number in the settings cell, then run every cell again (Runtime, then Run all).

- **A blunter cone:** `cone_half_angle = 30.0`, and compare its curve with the 20-degree cone's.
- **A finer grid:** `grid = 0.0625`. About four times slower; compare it with the converged curves.
- **A sphere-tipped needle:** in the needle's cell, use
  `needle = st.PunchTip(radius=mm(0.5), fillet=mm(0.5), length=mm(6.0))`, and in the clot's cell
  `tip_shape="softtissue.punch_tip"`.

## Adapting this to your own clot and tips

Start from the saved experiment, as the 3-D cell did, and declare again what is yours: the material (a record's name)
and the tip (its shape and dimensions). MIME replaces each declaration it took, warns you, and lists it in
`plan().replaced`; the new experiment records which one it was based on (its name, its input hash; later its DOI).

- **Your material.** Fit your rheology and compression data with the plugin's fitting harness
  (`python -m mime_softtissue.fit manifest.toml --out results/`, the columns in `docs/fit_spec.md`; it fits a neo-Hookean
  or fibre law), write the values into a record of your own (a JSON file with its licence, named in your plugin's
  `records.json`), and name it as `material`. The cell below uses the plugin's gel record as a stand-in.
- **Your toughness.** Calibrate Gc on one of your geometries and predict the others, with the split (which test
  calibrates, which ones test) written down and committed before any run, as the clot's tearing replication did.
- **Your tips.** A flat punch, a rounded or sphere-tipped needle (`st.PunchTip`), a cone or a needle with a conical point
  (`st.ConeTip`), or a headed tip (`st.HeadedTip`), with your dimensions; the block's `tip_shape` names its shape.
- **The contact's stiffness** is 100 times the material's Young's modulus over the cell size: attach the contact again
  when the material changes (it holds a number, not the record).
- **Not supported yet:** tips from a mesh or CAD file (with MIME's next development release); adhesion and the
  retrieval stroke in 3-D; rotation and drilling (with the plugin's first development release); a clot that varies from
  place to place.

In [ ]:
my = Experiment.based_on(saved("clot_puncture"), name="my_clot")
my_record = "softtissue.puncture_gel"  # your fitted record's name
my.add_body(Body(
    "clot",
    dynamics=st.GelBlock(radius=mm(5.0), height=mm(5.0), material=my_record, h=mm(0.125), fine=mm(1.0),
                         dimension="axisymmetric", tip_shape="softtissue.punch_tip"),
    initial=State(position=mm((0.0, 0.0, 0.0))),
))
my.add_body(Body(
    "tip",
    dynamics=RigidInertial(),
    parts={"tip": Part(st.PunchTip(radius=mm(0.5), fillet=mm(0.5), length=mm(6.0)))},  # a sphere-tipped needle
    initial=State(position=mm((0.0, 0.0, 5.0)), orientation=(1.0, 0.0, 0.0, 0.0),
                  velocity=m_per_s((0.0, 0.0, -speed)), angular_velocity=rad_per_s((0.0, 0.0, 0.0))),
))
with open(resolve(reference(my_record))) as f:
    my_young = json.load(f)["values"]["young_modulus"]
my.attach(st.TipContact(friction=0.0, adhesion=0.0, stiffness=100.0 * my_young / 0.125e-3), tip="tip", solid="clot")
for line in my.plan().replaced:
    print("replaced:", line[:160], "...")
my_folder = Path(tempfile.mkdtemp(prefix="my_clot_")) / "my_clot"
my.save(my_folder)  # runs its first steps, about a minute
print("saved", my_folder)